## tl;dr
本笔记只复算Git中的小型E35摘要，不重新读取原表、访问A/B或训练。当前映射快照覆盖25,437条TRAIN session；7个跨stage group的14条记录均在排除集合中。3次失败也都有范围外阶段排除记录，不能强行加进正式候选费用。映射链里的费用仅是历史引用代理，不是线上实际耗时。

## Context & Methods
### Key Assumptions
统计粒度分别为唯一session、source→session引用、action的全部已观察source均值。三者不混用。混合元数据中的VALIDATION只被过滤，不读取其runtime表；排除理由是旧流程声明，不等于资格认证。
输入是已独立审查reader产生的固定快照摘要，原始代码、合同和SHA清单都在仓库。此笔记不认证完整候选空间、独立调用身份或原始日志语义。
环境说明：本机没有nbformat、nbclient、ipykernel。代码单元可用标准Python按顺序核验；未宣称Jupyter内核执行。若已有Jupyter环境，可在仓库根运行：`python -m jupyter nbconvert --execute --to notebook --inplace experiment_overview/notebooks/E35_mapping_metadata_checks.ipynb`。

## Data
### 1. 读取有界摘要并校验SHA
只读取`data/manifests/ranking_v6_e35_mapping_summary_20261016_r1.json`（小于64 KiB）；不读取其中引用的任何源路径。

In [ ]:
import hashlib, json
from decimal import Decimal, localcontext
from pathlib import Path
repo = Path.cwd()
if not (repo / 'data/manifests').is_dir():
    repo = repo.parent.parent
summary_path = repo / 'data/manifests/ranking_v6_e35_mapping_summary_20261016_r1.json'
assert 0 < summary_path.stat().st_size < 65536
summary_bytes = summary_path.read_bytes()
assert hashlib.sha256(summary_bytes).hexdigest() == 'f2c2592d38f7658a35cfbe91710cf10d87e56099af9ae3e6be05ac93391ca0a2'
summary = json.loads(summary_bytes)
assert summary['real_training_release'] is False
print('Pinned bounded summary verified; no source tables opened.')

Pinned bounded summary verified; no source tables opened.


## Results
### 2. 核对覆盖、失败及三种计时口径
mapped与excluded应构成提供集合的完整分区，不能把excluded当作从未发生；失败费用仍留在账本中。

In [ ]:
partitions = summary['attempt_partitions']
assert sum(p['attempt_count'] for p in partitions.values()) == summary['provided_train_attempt_count']
assert partitions['both']['attempt_count'] == partitions['neither']['attempt_count'] == 0
with localcontext() as ctx:
    ctx.prec = 50
    all_wall = Decimal(summary['all_supplied_unique_wall_sum_s'])
    mapped_wall = Decimal(summary['mapped_unique_wall_sum_s'])
    edge_proxy = Decimal(summary['source_edge_incidence_historical_proxy_sum_s'])
    assert sum((Decimal(p['declared_wall_sum_s']) for p in partitions.values()), Decimal(0)) == all_wall
    assert mapped_wall == Decimal(partitions['mapped_only']['declared_wall_sum_s'])
    assert edge_proxy == Decimal(summary['all_source_actual_edge_sum_vector']['sum_s'])
    result = {'provided_sessions': summary['provided_train_attempt_count'], 'mapped_sessions': partitions['mapped_only']['attempt_count'], 'excluded_sessions': partitions['excluded_only']['attempt_count'], 'all_unique_declared_s': str(all_wall), 'mapped_unique_declared_s': str(mapped_wall), 'reference_proxy_s': str(edge_proxy), 'reference_minus_mapped_unique_s': str(edge_proxy-mapped_wall), 'excluded_failed_sessions': partitions['excluded_only']['outcome_counts']['FAILURE'], 'excluded_failed_declared_s': partitions['excluded_only']['declared_wall_by_outcome_s']['FAILURE']}
print(json.dumps(result, indent=2))

{
  "provided_sessions": 25437,
  "mapped_sessions": 17391,
  "excluded_sessions": 8046,
  "all_unique_declared_s": "252070.420",
  "mapped_unique_declared_s": "160758.330",
  "reference_proxy_s": "176806.570",
  "reference_minus_mapped_unique_s": "16048.240",
  "excluded_failed_sessions": 3,
  "excluded_failed_declared_s": "28.410"
}


### 3. 核对7个跨stage group的实际引用
使用完整引用集合统计，不从展示样例推断全部关联。

In [ ]:
groups = summary['mixed_stage_group_adjacency']
assert len(groups) == summary['mixed_attempt_stage_group_count'] == 7
assert summary['omitted_group_count'] == summary['omitted_member_count'] == summary['omitted_reference_count'] == 0
assert all(g['mapped_member_count'] == 0 and g['excluded_member_count'] == g['supplied_member_count'] for g in groups)
assert all(g['source_union_count'] == g['action_union_count'] == 0 for g in groups)
members = [m for g in groups for m in g['members']]
assert len(members) == 14 and all(m['exclusion_reason'] == 'OUT_OF_SCOPE_STAGE' for m in members)
assert summary['group_members_autoadded'] is False and summary['groups_split'] is False
print(json.dumps({'cross_stage_groups': len(groups), 'group_members': len(members), 'mapped_members': 0, 'declared_member_wall_s': str(sum((Decimal(m['declared_wall_s']) for m in members), Decimal(0))), 'candidate_cost_certified': summary['candidate_cost_certified']}, indent=2))

{
  "cross_stage_groups": 7,
  "group_members": 14,
  "mapped_members": 0,
  "declared_member_wall_s": "164.960",
  "candidate_cost_certified": false
}


### 4. 核对失败排除的聚焦证据
只读取另一个已绑定的小摘要，不读runtime原表。formal枚举匹配不是资格证书，排除理由仍是历史声明。

In [ ]:
failed_path = repo / 'data/manifests/ranking_v6_e35_failed_exclusion_summary_20261016_r1.json'
assert 0 < failed_path.stat().st_size < 65536
failed_bytes = failed_path.read_bytes()
assert hashlib.sha256(failed_bytes).hexdigest() == '98c816ebd8b3120df5888294b2f0778cbefac7064f73389b6efb6b015a995d0b'
failed = json.loads(failed_bytes)
assert failed['failed_or_timeout_count'] == 3 and failed['omitted_detail_count'] == 0
assert all(d['exclusion_reason'] == 'OUT_OF_SCOPE_STAGE' and d['stage_is_formal'] is False and d['exclusion_stage_mode_matches'] is True for d in failed['details'])
assert sum((Decimal(d['declared_wall_s']) for d in failed['details']), Decimal(0)) == Decimal('28.410')
assert failed['sessions_reassigned'] is False and failed['eligibility_inferred'] is False
print(json.dumps({'excluded_failed_sessions': 3, 'out_of_scope_stage_count': failed['by_exclusion_reason']['OUT_OF_SCOPE_STAGE']['count'], 'declared_failed_wall_s': failed['failed_timeout_declared_wall_sum_s'], 'costs_reassigned': failed['sessions_reassigned']}, indent=2))

{
  "excluded_failed_sessions": 3,
  "out_of_scope_stage_count": 3,
  "declared_failed_wall_s": "28.410",
  "costs_reassigned": false
}


## Takeaways
这7组在当前映射快照中没有进入正式候选引用，因此不能为了让E34的同stage假设通过而补边或拆组。全部提供记录已有mapped/excluded去向，但排除理由是否充分、source是否等于真实调用、候选空间是否完整，仍需独立证据。
245条session被多个source/action共享，故引用费用代理与唯一session秒数不同；这不直接等于数据重复或线上时间增加。3次失败都有范围外阶段排除记录，不能称为正式候选已证实漏账，也不能声称完整历史失败开销已经认证。
详细结论与后续最小修复见`experiment_overview/entries/E35_mapping_metadata_and_cost_scope.md`。不要从本笔记启动训练、调参、BLIND或LSF/Tessent。